# Sepsis Prediction (6-12h Horizon)

Predict whether a patient will develop sepsis within the next 6-12 ICU hours.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# Config
ROOT = Path.cwd().resolve()
if ROOT.name != "safe_dss":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "preprocessing" / "imputed_dataset.csv"
OUT_DIR = ROOT / "analysis" / "outputs" / "sepsis_6_12h"
EARLY_WARNING_ONLY = True

print("Data:", DATA_PATH)
print("Output:", OUT_DIR)

In [ ]:
def load_and_clean(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for c in df.columns:
        if c == "patient_id":
            continue
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if "EtCO2" in df.columns and df["EtCO2"].isna().all():
        df = df.drop(columns=["EtCO2"])

    for c in ("Unit1", "Unit2"):
        if c in df.columns and df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    return df


def add_pattern_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["patient_id", "ICULOS"]).copy()
    g = df.groupby("patient_id", sort=False)

    roll_cols = ["MAP", "HR", "Lactate", "Resp", "WBC", "Temp", "Creatinine", "Glucose"]
    roll_cols = [c for c in roll_cols if c in df.columns]

    for c in roll_cols:
        df[f"{c}_roll3"] = g[c].transform(lambda s: s.rolling(window=3, min_periods=1).mean())

    for c in ["MAP", "Lactate", "HR"]:
        if c in df.columns:
            df[f"{c}_delta1"] = g[c].diff().fillna(0.0)

    return df


def add_horizon_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["patient_id", "ICULOS"]).copy()
    g = df.groupby("patient_id", sort=False)["SepsisLabel"]

    fut = pd.concat([g.shift(-k).rename(f"_f{k}") for k in range(6, 13)], axis=1)
    df["y_onset_6_12h"] = fut.max(axis=1, skipna=False)
    df["_future_hours_ok"] = fut.notna().all(axis=1)
    return df


def build_matrix(df: pd.DataFrame):
    drop_cols = {"patient_id", "SepsisLabel", "y_onset_6_12h", "_future_hours_ok"}
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df["y_onset_6_12h"].astype(int).values
    groups = df["patient_id"].values
    return X, y, groups

In [ ]:
df = load_and_clean(DATA_PATH)
df = add_pattern_features(df)
df = add_horizon_labels(df)
df = df[df["_future_hours_ok"]].copy()

if EARLY_WARNING_ONLY:
    df = df[df["SepsisLabel"] == 0].copy()
    cohort_note = "early_warning_cohort (SepsisLabel == 0 at t)"
else:
    cohort_note = "all rows with full future window"

X, y, groups = build_matrix(df)
feature_names = list(X.columns)

print("Cohort:", cohort_note)
print(f"Rows: {len(df):,}")
print(f"Positive rate (y): {y.mean():.4f}")
print("N features:", len(feature_names))

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []
coef_accum = []

for fold, (tr, va) in enumerate(sgkf.split(X, y, groups)):
    X_tr, X_va = X.iloc[tr], X.iloc[va]
    y_tr, y_va = y[tr], y[va]

    pre = ColumnTransformer([
        ("num", StandardScaler(), feature_names),
    ], remainder="drop")

    logit = Pipeline([
        ("prep", pre),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")),
    ])
    logit.fit(X_tr, y_tr)
    p_logit = logit.predict_proba(X_va)[:, 1]

    fold_metrics.append({
        "fold": fold,
        "model": "logistic_l2",
        "auroc": float(roc_auc_score(y_va, p_logit)),
        "auprc": float(average_precision_score(y_va, p_logit)),
    })
    coef_accum.append(np.abs(logit.named_steps["clf"].coef_.ravel()))

    hgb = HistGradientBoostingClassifier(
        max_depth=6,
        max_iter=200,
        learning_rate=0.06,
        random_state=42,
        class_weight="balanced",
    )
    hgb.fit(X_tr, y_tr)
    p_hgb = hgb.predict_proba(X_va)[:, 1]

    fold_metrics.append({
        "fold": fold,
        "model": "hist_gbrt",
        "auroc": float(roc_auc_score(y_va, p_hgb)),
        "auprc": float(average_precision_score(y_va, p_hgb)),
    })

metrics_df = pd.DataFrame(fold_metrics)
metrics_summary = metrics_df.groupby("model")[["auroc", "auprc"]].agg(["mean", "std"])
metrics_summary

In [ ]:
# Final holdout split for reporting + importance
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_tr, idx_te = next(gss.split(X, y, groups))

X_tr, X_te = X.iloc[idx_tr], X.iloc[idx_te]
y_tr, y_te = y[idx_tr], y[idx_te]

pre = ColumnTransformer([
    ("num", StandardScaler(), feature_names),
], remainder="drop")

final_logit = Pipeline([
    ("prep", pre),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")),
])
final_logit.fit(X_tr, y_tr)
p_logit_te = final_logit.predict_proba(X_te)[:, 1]

hgb_final = HistGradientBoostingClassifier(
    max_depth=6,
    max_iter=200,
    learning_rate=0.06,
    random_state=42,
    class_weight="balanced",
)
hgb_final.fit(X_tr, y_tr)
p_hgb_te = hgb_final.predict_proba(X_te)[:, 1]

print("Holdout Logistic AUROC:", round(roc_auc_score(y_te, p_logit_te), 4))
print("Holdout Logistic AUPRC:", round(average_precision_score(y_te, p_logit_te), 4))
print("Holdout HGB AUROC:", round(roc_auc_score(y_te, p_hgb_te), 4))
print("Holdout HGB AUPRC:", round(average_precision_score(y_te, p_hgb_te), 4))

In [ ]:
# Importance outputs
coef_mean = np.mean(coef_accum, axis=0)
logit_imp = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_coef_cv": coef_mean,
}).sort_values("mean_abs_coef_cv", ascending=False)

rng = np.random.RandomState(42)
n_perm = min(8000, len(X_te))
perm_pos = rng.choice(len(X_te), size=n_perm, replace=False)
X_perm = X_te.iloc[perm_pos]
y_perm = y_te[perm_pos]

perm_res = permutation_importance(
    hgb_final,
    X_perm,
    y_perm,
    n_repeats=5,
    random_state=42,
    scoring="roc_auc",
    n_jobs=1,
)
perm_imp = pd.DataFrame({
    "feature": feature_names,
    "delta_roc_auc_mean": perm_res.importances_mean,
}).sort_values("delta_roc_auc_mean", ascending=False)

# Plot top features instead of only listing tables
k = 15
plot_logit = logit_imp.head(k).sort_values("mean_abs_coef_cv", ascending=True)
plot_perm = perm_imp.head(k).sort_values("delta_roc_auc_mean", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 8), constrained_layout=True)

axes[0].barh(plot_logit["feature"], plot_logit["mean_abs_coef_cv"], color="#1f77b4")
axes[0].set_title("Logistic importance (|coef|, CV mean)")
axes[0].set_xlabel("Mean absolute coefficient")
axes[0].set_ylabel("Feature")

axes[1].barh(plot_perm["feature"], plot_perm["delta_roc_auc_mean"], color="#ff7f0e")
axes[1].set_title("HGB permutation importance")
axes[1].set_xlabel("Mean AUROC drop after permutation")
axes[1].set_ylabel("Feature")

plt.show()

print("Top logistic features (table):")
display(logit_imp.head(k))
print("Top HGB permutation features (table):")
display(perm_imp.head(k))

### b(iii) Patient profiles (interpretation from Stage 1)

The repository does not contain true 30-day post-discharge readmissions or ICD-coded comorbidities. **Profile insight** is drawn from **Stage 1** here: demographics (`Age`, `Gender`), timing (`ICULOS`, `HospAdmTime`), ICU mapping (`Unit1`–`Unit2`), and physiology/lab trajectories. A **Random Forest** on the **same 6–12h onset target** gives nonlinear risk estimates; **Gini-based feature importances** from the forest (bar plot + CSV) summarize which inputs the model relies on most.

---

Run the next cell after the holdout models (`X_tr`, `X_te`, `y_tr`, `y_te`, `feature_names`) are defined.

In [ ]:
# Random Forest (Stage 1) + feature importance (Gini)
from sklearn.ensemble import RandomForestClassifier

rf_stage1 = RandomForestClassifier(
    n_estimators=400,
    max_depth=16,
    min_samples_leaf=15,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
rf_stage1.fit(X_tr, y_tr)
p_rf_te = rf_stage1.predict_proba(X_te)[:, 1]
print("Stage-1 RF holdout AUROC:", round(roc_auc_score(y_te, p_rf_te), 4))
print("Stage-1 RF holdout AUPRC:", round(average_precision_score(y_te, p_rf_te), 4))

rf_imp1 = (
    pd.DataFrame({"feature": feature_names, "importance": rf_stage1.feature_importances_})
    .sort_values("importance", ascending=True)
)
k_rf = min(20, len(rf_imp1))
plot_rf = rf_imp1.tail(k_rf)
plt.figure(figsize=(9, 7))
plt.barh(plot_rf["feature"], plot_rf["importance"], color="#2ca02c")
plt.xlabel("Gini importance (mean decrease impurity)")
plt.title("Stage 1 RF: feature importance (6–12h sepsis onset)")
plt.tight_layout()
plt.show()

OUT_DIR.mkdir(parents=True, exist_ok=True)
rf_imp1.sort_values("importance", ascending=False).to_csv(OUT_DIR / "rf_feature_importance_stage1.csv", index=False)
print("Saved", OUT_DIR / "rf_feature_importance_stage1.csv")

In [ ]:
# Persist results for assignment artifacts
OUT_DIR.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(OUT_DIR / "cv_metrics_by_fold.csv", index=False)
logit_imp.to_csv(OUT_DIR / "logistic_coefficient_magnitude_cv.csv", index=False)
perm_imp.to_csv(OUT_DIR / "permutation_importance_hist_gbrt_holdout.csv", index=False)

summary = {
    "task": "Binary onset of sepsis in hours t+6..t+12",
    "cohort": cohort_note,
    "n_rows": int(len(df)),
    "positive_rate": float(y.mean()),
    "holdout_auroc_logistic": float(roc_auc_score(y_te, p_logit_te)),
    "holdout_auprc_logistic": float(average_precision_score(y_te, p_logit_te)),
    "holdout_auroc_hist_gbrt": float(roc_auc_score(y_te, p_hgb_te)),
    "holdout_auprc_hist_gbrt": float(average_precision_score(y_te, p_hgb_te)),
    "holdout_report_hist_gbrt": classification_report(y_te, hgb_final.predict(X_te), digits=4, output_dict=True),
}
try:
    summary["holdout_auroc_rf_stage1"] = float(roc_auc_score(y_te, p_rf_te))
    summary["holdout_auprc_rf_stage1"] = float(average_precision_score(y_te, p_rf_te))
except NameError:
    pass

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved outputs to", OUT_DIR)